# 03 - Model Training, Tuning, and Evaluation

## Imports

In [27]:
import warnings
warnings.filterwarnings('ignore') # ignoring warnings for sake of space - don't do this normally! 


import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error
from sklearn.datasets import load_diabetes
import pandas as pd 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge

# Make sure we get outputs from a probabilitistic model each time (for reproducibility) 
np.random.seed(1515)

## How can we make Generalizable Models?

### Data Splits

In our previous example, we covered how there was a difference in performance between Period 1/3 data and Period 5 data for both the good and bad models. 

If we follow the path we took for the good model specifically, we built our model on the good data from Period 3, and tested in on Period 5 data. This was done to ensure our model performed well not just on the Period 3 data, but another dataset (Period 5). 

In general, we only have one dataset to work with. So, we have to split our existing data into the following categories:

1. **Training**: this is the data we build our original model on
2. **Validation**: this is the data we alter our model on to see what the best performance is
3. **Test**: this is the data we perform a final test on to ensure generalizability

A common way to split this data is 70/15/15, which represents the percentage of total data in the training/validation/test splits. 


Let's use a sample dataset to show what this could look like! Note that typically, we would be doing some further analysis to better understand our data, but we will save that for when we cover Computer Vision/Perception. This is more just to show how you evaluate model results in general.

In [32]:
# read in diabetes dataset from Seaborn 
diabetes = load_diabetes()

# store as pandas DataFrame
df = pd.DataFrame(data=diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target
df.head(5)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


This is a premade dataset that shows diabetes progression along with age, sex, BMI, Blood Pressure, and 6 Serum measurements. 'target' is the measure of the diabetes progression. 

Let's make sure our dataset looks to be in order: 

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
 10  target  442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB


We have 442 rows, none of which are null (or no value), so we can proceed to splitting our data:

In [38]:
X = df.drop(columns=['target']) # X is all of our independent/predictor variables
y = df['target'] # y is our target/outcome variable 

# first split: Get training data
# second split: split remaining into validation and test data

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.3, random_state = 1515)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, random_state = 1515)

print("\nSplit sizes:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


Split sizes:
Train: (309, 10) Val: (66, 10) Test: (67, 10)


We have split our data successfully! Note that the first number in each pair of parentheses is the number of subjects/rows, the second is the number of features/columns. Now, how do we go about building our model? 

### General Model Selection

For our previous example regarding AP Calculus scores, we used RMSE to determine which model was better. However, we didn't really have a systematic way of picking a model; we simply used a line or polynomial function that was picked for us, and did a comparison. 

Naturally, as our model gets more complicated, we are going to want to test out different variations of it to determine which one performs the best. And by performs the best, we mean that it has the lowest RMSE for the test dataset. 

So, what could some different variations for this problem look like? Well, we know from our previous example that attempting to use a different number of parameters/different complexity was one option. But we remember that we didn't really have a quantitative method of picking a good number of parameters; we sort of just picked something based on an existing function and by eyeballing it. 

The process of picking a number of parameters that minimizes loss whilst also preventing overfitting is known as **regularization**.

### Regularization

Regularization prevents overfitting by adding a penalty term to our loss equation. This will increase the loss of a model as it includes more parameters. Therefore, the model is forced to only keep the most important variables in terms of understanding the patterns in the data we want to understand. 

As we saw in the previous model, less parameters = less overfitting because our model is not capturing training data noise! 

Regularization takes on this general form for our loss equation: 

$RMSE = \lambda\sqrt{{\frac{1}{n}\sum^n_{i=1}(\hat{y_i} - y_i)^2}}$

Where the lambda ($\lambda$) term represents the penalty we apply. There are a few different penalties we can apply, with L1/L2 or a combination of both (Elastic Net) as the most common option. Again, we will avoid looking at the mathematical justification of these penalties and instead focus on how we can find the figure that optimizes our equation. 

### Other Tuning Parameters

There are other parameters to tune depending on the type of model you choose to build. Instead of going through an exhaustive list of possibilities, we will run an example so you can see what one case of this looks like. Remember, you can always look these things up!

The most important takeaway here is an understanding of *how* to choose the right hyperparameters.

The first step is to define a grid of the possible parameters you want to tune. Once you have that set of parameters, you choose to either run through all of the combinations (a grid search), or randomly select a subset of the combinations (random search), and you measure model performance. Let's walk through the most common form of this process, known as **cross validation**.

### Cross-Validation

Cross validation is the general technique of taking our training data, splitting it into folds, and running variations of our model on these different folds--where some folds serve as our validation data--and we then pick the model with the best performance on that validation set.

There are several types of cross-validation, the most common of which is k-fold cross validation, where we divide our dataset into k different folds, where the remaining k-1 folds are training data (the kth is validation data), and we rotate through all combinations of training/validation splits and get our model performance. We pick the model configuration that gives us the best performance.

Now that we've established a rough outline of what this looks like, let's get to coding! 

## Training/Evaluation Loop

### Baseline Model

When creating models, it is always useful to have a baseline. Your baseline model is going to have poor performance - this is by design. For instance, in our situation, since we are performing a **regression**--where we use different parameters to predict the value of a variable of interest--our baseline is usually just going to be the average of the outcome variable, in our training data. The average should be right around "middle of the road" performance in comparison to any possible model we can train, so it gives us a good baseline: if our trained model is better, we know we are on the right track. If it is worse, we know we have a *really* bad model on our hands. 

In this case, it'll be the average of our diabetes progression. Note that we train/fit the baseline model on our training data, as per usual, but we do the predictions on our validation data for an even comparison with the tuned models. 

Let's define this below: 

In [61]:
# define our baseline model 

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)
print(f'Baseline RMSE: {round(root_mean_squared_error(y_val, baseline_pred), 2)}')

Baseline RMSE: 72.89


Now we have a source of comparison for our tuned models - let's get to that. 

### Hyperparameter Tuning

Hyperparameter tuning is the formal name of the process we have been talking about - picking the hyperparameters that give us optimal performance. What makes hyperparameter tuning so useful is that it is essentially a structured guess and check. We create our grid of parameters, and pick the one that works best! 

Let's walk through each of those steps:

In [73]:
# define our regression pipeline 

ridge_poly = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", Ridge())  
])

To quickly explain what is going up, we are defining a nonlinear regression (more than one parameter, like our overfit bad model). We are performing our regularization with L2/Ridge Regression, which works on penalizing what is known as L2 loss. This is the most common form of regularization for regression. 

In [76]:
# combine our train/validation (since we split it back up for cross-validation) 

X_tune = pd.concat([X_train, X_val], axis=0)
y_tune = pd.concat([y_train, y_val], axis=0)

We combine our training and validation data together for tuning since we will be creating the validation data via the cross fold validation for tuning.

In [79]:
# create the k-fold cross-validation

cv = KFold(n_splits=5, shuffle=True, random_state=1515)

We create 5 folds to test, which is a pretty common number to pick. We also shuffle our folds for randomness and set a random state so we get reproducible outputs (so we can recreate what happens more easily if we share our code with others)

In [84]:
# Parameter grid / search space

ridge_params = {
    "poly__degree": [1, 2, 3], # degree of polynomial for our regression 
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0], # penalty parameter for Ridge 
}

We set our grid space with both the polynomial degree of our regression and the penalty parameter for Ridge. Here, we picked some rather arbitrary values for the sake of our example, but typically you want to find some well-established values, or values that would fit the data you have the best. This could take some extensive research or consulting with ChatGPT.

In [92]:
# Grid Search of parameters to try every combination 

ridge_search = GridSearchCV(
    ridge_poly,
    param_grid=ridge_params,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1
)
ridge_search.fit(X_tune, y_tune)

print("\nBest Ridge+Poly params:", ridge_search.best_params_)
print("Best Ridge+Poly CV RMSE:", -ridge_search.best_score_)


Best Ridge+Poly params: {'model__alpha': 1.0, 'poly__degree': 1}
Best Ridge+Poly CV RMSE: 56.01832362440963


Since we had only a few combinations, it is fine for us to run a grid search, where we run every combination of parameters. However, if we have a lot of different combinations, it is more efficient to run a random search where we randomly pick several combinations to test.

Looking at our results, we have the parameters that gave us the best score in the loop, which we will put in our final model to run on the test data. Also, our RMSE is significantly lower than that of our baseline, suggesting we have built a strong model. 

## Final Evaluation: Performance on Test Data

Now, we finally get to test our model on the test dataset, which will show if our model is generalizable. 

In [97]:
# running best model on  

final_model = ridge_search.best_estimator_ # get best model configuration 
final_model.fit(X_tune, y_tune) # fit model to train+val data 

final_pred = final_model.predict(X_test)
print(f'Test RMSE: {round(root_mean_squared_error(y_test, final_pred), 2)}')

Test RMSE: 46.18


Great! As we can see, our test RMSE is lower than both our training and validation RMSEs, which tells us our model generalizes well. 

We have successfully walked through the evaluation of a model via training and tuning it to produce a generalizable model. This type of technique will translate to any type of model you want to train, including the models we will be building for object detection. 